In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split



In [ ]:
dataset = pd.read_csv('Housing.csv')
dataset.head()


In [ ]:
label_encoder = LabelEncoder()
dataset['mainroad_code'] = label_encoder.fit_transform(dataset['mainroad'])
dataset['guestroom_code'] = label_encoder.fit_transform(dataset['guestroom'])
dataset['basement_code'] = label_encoder.fit_transform(dataset['basement']) 
dataset['hotwaterheating_code'] = label_encoder.fit_transform(dataset['hotwaterheating'])
dataset['airconditioning_code'] = label_encoder.fit_transform(dataset['airconditioning'])
dataset['prefarea_code'] = label_encoder.fit_transform(dataset['prefarea'])
dataset['furnishingstatus_code'] = label_encoder.fit_transform(dataset['furnishingstatus'])

dataset.tail()

In [ ]:
# Detect outliers in numeric columns using the IQR method
# Only use numeric columns that are not encoded (we exclude the label-encoded *_code columns).
numeric_cols = dataset.select_dtypes(include=[np.number]).columns
numeric_cols = [c for c in numeric_cols if not c.endswith("_code")]

Q1 = dataset[numeric_cols].quantile(0.25)
Q3 = dataset[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

outlier_mask = (dataset[numeric_cols] < (Q1 - 1.5 * IQR)) | (dataset[numeric_cols] > (Q3 + 1.5 * IQR))
outlier_counts = outlier_mask.sum().sort_values(ascending=False)

print("Numeric columns with outliers (count of rows with at least one outlier in that column):")
print(outlier_counts[outlier_counts > 0])

# Show a small sample of rows that have any outlier
has_outlier = outlier_mask.any(axis=1)
print(f"\nTotal rows with any numeric outlier: {has_outlier.sum()}")
dataset.loc[has_outlier].head()

# Remove rows with any numeric outliers from the dataset
dataset = dataset.loc[~has_outlier].reset_index(drop=True)
print(f"Rows remaining after dropping outliers: {dataset.shape[0]}")

# Visualize the numeric columns that have outliers
import matplotlib.pyplot as plt
import seaborn as sns

cols_with_outliers = outlier_counts[outlier_counts > 0].index.tolist()
print(f"\nColumns detected with outliers: {cols_with_outliers}")

if cols_with_outliers:
    # Plot a boxplot for each column with outliers
    fig, axes = plt.subplots(nrows=len(cols_with_outliers), ncols=1, figsize=(8, 4 * len(cols_with_outliers)))
    if len(cols_with_outliers) == 1:
        axes = [axes]
    for ax, col in zip(axes, cols_with_outliers):
        sns.boxplot(x=dataset[col], ax=ax)
        ax.set_title(f"Boxplot: {col}")
        ax.set_xlabel(col)
    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns with outliers to visualize.")

In [ ]:
# Create correlation heatmap excluding label-encoded features
plt.figure(figsize=(12, 10))
numeric_data = dataset.select_dtypes(include=[np.number])
# Exclude label-encoded columns (those ending with "_code")
numeric_data = numeric_data[[col for col in numeric_data.columns if not col.endswith("_code")]]
correlation_matrix = numeric_data.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap - Housing Dataset (Excluding Encoded Features)')
plt.tight_layout()
plt.show()

In [ ]:
# Standardize the area feature before training
area_scaler = StandardScaler()
dataset['area_scaled'] = area_scaler.fit_transform(dataset[['area']])

feature_list = [
    'area_scaled', 'bedrooms', 'bathrooms', 'stories', 'parking',
    'mainroad_code', 'guestroom_code', 'basement_code',
    'hotwaterheating_code', 'airconditioning_code',
    'prefarea_code', 'furnishingstatus_code'
]

X = dataset[feature_list]
y = dataset['price']
print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
dataset.head()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=None)

print(f'Training features shape: {X_train.shape}')
print(f'Training target shape: {y_train.shape}')
print(f'Test features shape: {X_test.shape}')
print(f'Test target shape: {y_test.shape}')

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Random Forest does not require feature scaling, so we use unscaled data
# Create and train the Random Forest model
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# Predict on the test set
predictions = model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print('Random Forest evaluation:')
print(f'Mean Squared Error:       {mse:.2f}')
print(f'Root Mean Squared Error:  {rmse:.2f}')
print(f'Mean Absolute Error:      {mae:.2f}')
print(f'R² score:                  {r2:.4f}')

# Feature importance
importance_df = pd.DataFrame({
    'Feature': feature_list,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='steelblue')
plt.xlabel('Importance')
plt.title('Random Forest — Feature importances')
plt.tight_layout()
plt.show()


In [ ]:
# Predict the price for a new house record
# new_house values must follow the feature_list order:
# ['area_scaled', 'bedrooms', 'bathrooms', 'stories', 'parking',
#  'mainroad_code', 'guestroom_code', 'basement_code',
#  'hotwaterheating_code', 'airconditioning_code',
#  'prefarea_code', 'furnishingstatus_code']

raw_new_house = {
    'area': 6000,
    'bedrooms': 4,
    'bathrooms': 1,
    'stories': 2,
    'parking': 1,
    'mainroad_code': 1,
    'guestroom_code': 0,
    'basement_code': 1,
    'hotwaterheating_code': 0,
    'airconditioning_code': 0,
    'prefarea_code': 0,
    'furnishingstatus_code': 1
}

new_house_df = pd.DataFrame([raw_new_house])
new_house_df['area_scaled'] = area_scaler.transform(new_house_df[['area']])
new_house_df = new_house_df[feature_list]

# Random Forest does not require scaling
predicted_price = model.predict(new_house_df)[0]
print(f'Predicted price for the new house: {predicted_price:.2f}')

# Plot actual vs predicted prices for the test set
results_df = pd.DataFrame({
    'Actual': y_test.reset_index(drop=True),
    'Predicted': predictions
})

plt.figure(figsize=(8, 6))
sns.scatterplot(x='Actual', y='Predicted', data=results_df, alpha=0.7)
plt.plot([results_df['Actual'].min(), results_df['Actual'].max()],
         [results_df['Actual'].min(), results_df['Actual'].max()],
         color='red', linestyle='--', label='Perfect prediction')
plt.title('Random Forest — Actual vs Predicted House Prices')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.legend()
plt.tight_layout()
plt.show()
